In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pickle

np.random.seed(42)

# === PARAMETRY — zmień tutaj ===
N_NORMAL = 2000      # liczba normalnych transakcji
N_FRAUD  = 100       # liczba fraudów
# ===============================

# Normalne transakcje
normal = pd.DataFrame({
    'amount': np.random.lognormal(5, 1, N_NORMAL).clip(5, 5000),
    'is_electronics': np.random.binomial(1, 0.3, N_NORMAL),
    'tx_per_minute': np.random.poisson(3, N_NORMAL),
    'fraud': 0
})


# Fraudy
fraud = pd.DataFrame({
    'amount': np.random.uniform(2000, 9000, N_FRAUD),
    'is_electronics': np.random.binomial(1, 0.7, N_FRAUD),
    'tx_per_minute': np.random.poisson(8, N_FRAUD),
    'fraud': 1
})

df = pd.concat([normal, fraud], ignore_index=True).sample(frac=1, random_state=42)
print(f"Dataset: {len(df)} wierszy, fraud rate: {df['fraud'].mean():.1%}")

Dataset: 2100 wierszy, fraud rate: 4.8%


In [2]:
features = ['amount', 'is_electronics', 'tx_per_minute']
X = df[features]
y = df['fraud']

# 1. train_test_split (80/20, stratify=y)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 2. RandomForestClassifier(100)
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)
model.fit(X_train, y_train)

# 3. classification_report
y_pred = model.predict(X_test)
print("\nClassification report:")
print(classification_report(y_test, y_pred))

# 4. pickle.dump do 'fraud_model.pkl'
with open('fraud_model.pkl', 'wb') as f:
    pickle.dump(model, f)

print("\nModel zapisany do pliku fraud_model.pkl")


Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       400
           1       1.00      1.00      1.00        20

    accuracy                           1.00       420
   macro avg       1.00      1.00      1.00       420
weighted avg       1.00      1.00      1.00       420


Model zapisany do pliku fraud_model.pkl


In [3]:
import os
print(os.listdir())

['.ipynb_checkpoints', '.notremove', 'fraud_api.py', 'fraud_model.pkl', 'kafka', 'Lab1', 'Lab2', 'ml_consumer.py', '__pycache__']


In [11]:
%%file fraud_api.py
from fastapi import FastAPI
from pydantic import BaseModel
import pickle
import pandas as pd

app = FastAPI(title="Fraud Detection API")

with open("fraud_model.pkl", "rb") as f:
    model = pickle.load(f)

FEATURES = ['amount', 'is_electronics', 'tx_per_minute']

class Transaction(BaseModel):
    amount: float
    is_electronics: int
    tx_per_minute: int

@app.post("/score")
def score(tx: Transaction):
    X = pd.DataFrame(
        [[tx.amount, tx.is_electronics, tx.tx_per_minute]],
        columns=FEATURES
    )

    pred = model.predict(X)[0]
    proba = model.predict_proba(X)[0][1]

    return {
        "is_fraud": bool(pred),
        "fraud_probability": float(proba)
    }

Overwriting fraud_api.py


In [1]:
import requests

# Test normalna
r = requests.post(
    "http://localhost:8001/score",
    json={
        "amount": 150,
        "hour": 14,
        "is_electronics": 0,
        "tx_per_minute": 3
    }
)
print("Normalna:", r.json())


# Test podejrzana
r2 = requests.post(
    "http://localhost:8001/score",
    json={
        "amount": 5500,
        "hour": 2,
        "is_electronics": 1,
        "tx_per_minute": 12
    }
)
print("Podejrzana:", r2.json())

ConnectionError: HTTPConnectionPool(host='localhost', port=8001): Max retries exceeded with url: /score (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x70fd1ec03fd0>: Failed to establish a new connection: [Errno 111] Connection refused'))

In [10]:
%%file ml_consumer.py
from kafka import KafkaConsumer, KafkaProducer
from datetime import datetime
import json
import requests

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='ml-scoring-v2',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

alert_producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

API_URL = "http://localhost:8001/score"

for msg in consumer:
    tx = msg.value

    try:
        # --- 1. Feature engineering ---
        amount = tx.get("amount", 0)
        is_electronics = tx.get("is_electronics", 0)

        # timestamp → hour (opcjonalnie, do debug/logów)
        timestamp = tx.get("timestamp")
        if timestamp:
            hour = datetime.fromisoformat(timestamp).hour
        else:
            hour = None

        # uproszczenie zgodnie z zadaniem
        tx_per_minute = 5

        features = {
            "amount": amount,
            "is_electronics": is_electronics,
            "tx_per_minute": tx_per_minute
        }

        # --- 2. Call API ---
        response = requests.post(API_URL, json=features)

        result = response.json()

        # --- 3. Jeśli fraud → alert ---
        if result.get("is_fraud"):
            alert = {
                "timestamp": datetime.utcnow().isoformat(),
                "transaction": tx,
                "fraud_probability": result.get("fraud_probability")
            }

            alert_producer.send('alerts', alert)

            print("ALERT:", alert)

        else:
            print("OK:", result)
    except Exception as e:
        print("ERROR:", e)

Overwriting ml_consumer.py


In [3]:
import requests

try:
    r = requests.get('http://localhost:8001/health', timeout=2)
    print('API działa:', r.json())
except Exception as e:
    print('API niedostępne — uruchom uvicorn w terminalu:', e)

API działa: {'status': 'ok'}
